<a href="https://colab.research.google.com/github/SebastianZV010/ChatMultipleClient/blob/main/transformer_proyect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pkg_resources, os, warnings
warnings.filterwarnings("ignore")
IN_COLAB = 'google-colab' in [p.key for p in pkg_resources.working_set]
os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # reproducibilidad + evitar warnings

if IN_COLAB:
    !pip -q install datasets scikit-learn 'transformers[torch]' accelerate

/tmp/ipython-input-1867206488.py:1: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources, os, warnings


In [2]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DISPOSITIVO:", DEVICE)


DISPOSITIVO: cpu


Carga de datos

In [5]:
# Dataset español listo para usar (5 clases)
from datasets import load_dataset
raw = load_dataset("SetFit/amazon_reviews_multi_es")  # {'train','validation','test'}

# Inspección rápida
print(raw)
print(raw["train"][0])           # {'id': 'es_...', 'text': '...', 'label': 0..4, 'label_text': '...'}


# Para mantener compatibilidad con tu notebook previo:
TR_TEXTS = raw["train"]["text"];    TR_Y = raw["train"]["label"]      # o 'label3' si reagrupaste
VAL_TEXTS = raw["validation"]["text"]; VAL_Y = raw["validation"]["label"]
TE_TEXTS  = raw["test"]["text"];     TE_Y  = raw["test"]["label"]

# Sanity check
print(len(TR_TEXTS), len(VAL_TEXTS), len(TE_TEXTS))
print(set(TR_Y), set(VAL_Y), set(TE_Y))


README.md:   0%|          | 0.00/310 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/43.8M [00:00<?, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5000
    })
})
{'id': 'es_0491108', 'text': 'Nada bueno se me fue ka pantalla en menos de 8 meses y no he recibido respuesta del fabricante', 'label': 0, 'label_text': '0'}
200000 5000 5000
{0, 1, 2, 3, 4} {0, 1, 2, 3, 4} {0, 1, 2, 3, 4}


Split

In [10]:
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np

def stratified_subsplit(dataset, text_key="text", y_key="label",
                        train_size=12000, val_size=2000, test_size=2000, seed=42):
    """
    Crea subconjuntos estratificados reproducibles a partir de un dataset HF con splits.
    Funciona con SetFit/amazon_reviews_multi_es (text, label).
    """
    texts, labels = [], []
    for split_name in ["train", "validation", "test"]:
        for r in dataset[split_name]:
            texts.append(r[text_key])
            labels.append(r[y_key])

    texts  = np.array(texts)
    labels = np.array(labels)

    total = train_size + val_size + test_size
    assert total <= len(texts), "El subconjunto pedido es mayor al tamaño total disponible"

    sss1 = StratifiedShuffleSplit(n_splits=1, train_size=total, random_state=seed)
    idx_all, _ = next(sss1.split(texts, labels))
    X_all, y_all = texts[idx_all], labels[idx_all]

    sss2 = StratifiedShuffleSplit(n_splits=1, train_size=train_size, random_state=seed)
    idx_tr, idx_temp = next(sss2.split(X_all, y_all))
    X_tr, y_tr = X_all[idx_tr], y_all[idx_tr]
    X_temp, y_temp = X_all[idx_temp], y_all[idx_temp]

    sss3 = StratifiedShuffleSplit(n_splits=1, train_size=val_size, random_state=seed)
    idx_val, idx_test = next(sss3.split(X_temp, y_temp))
    X_val, y_val = X_temp[idx_val], y_temp[idx_val]
    X_test, y_test = X_temp[idx_test], y_temp[idx_test]

    return (X_tr.tolist(), y_tr.tolist()), (X_val.tolist(), y_val.tolist()), (X_test.tolist(), y_test.tolist())

# 👇 Llámalo así (OJO: keys correctas y 5 clases)
(TR_TEXTS, TR_Y), (VAL_TEXTS, VAL_Y), (TE_TEXTS, TE_Y) = stratified_subsplit(
    raw, text_key="text", y_key="label", train_size=12000, val_size=2000, test_size=2000, seed=SEED
)

print(f"Tamaños -> train:{len(TR_TEXTS)} val:{len(VAL_TEXTS)} test:{len(TE_TEXTS)}")

# Distribución por clase (0..4 en este dataset)
cls, cnt = np.unique(TR_Y, return_counts=True)
print("Distribución train:", dict(zip(cls.tolist(), cnt.tolist())))


Tamaños -> train:12000 val:2000 test:2000
Distribución train: {0: 2400, 1: 2400, 2: 2400, 3: 2400, 4: 2400}


BASELINE: TF-IDF + MLP

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import torch
import torch.nn as nn

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2
)
X_train = tfidf.fit_transform(TR_TEXTS).astype(np.float32)
X_val   = tfidf.transform(VAL_TEXTS).astype(np.float32)
X_test  = tfidf.transform(TE_TEXTS).astype(np.float32)

y_train = np.array(TR_Y, dtype=np.int64)
y_val   = np.array(VAL_Y, dtype=np.int64)
y_test  = np.array(TE_Y, dtype=np.int64)

# >>> FIX: detecta automáticamente el número de clases (en este dataset son 5: 0..4)
NUM_CLASSES = len(set(y_train.tolist()) | set(y_val.tolist()) | set(y_test.tolist()))
print("NUM_CLASSES =", NUM_CLASSES)
assert y_train.max() <= NUM_CLASSES - 1 and y_val.max() <= NUM_CLASSES - 1 and y_test.max() <= NUM_CLASSES - 1

class MLPBaseline(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.net(x)

def iter_loader_sparse(X, y, batch_size=256, shuffle=True, device=DEVICE):
    idx = np.arange(X.shape[0])
    if shuffle:
        rng = np.random.default_rng(SEED)
        rng.shuffle(idx)
    for i in range(0, len(idx), batch_size):
        j = idx[i:i+batch_size]
        # Convertimos a denso solo por batch para ahorrar RAM
        Xb = torch.tensor(X[j].toarray(), device=device)
        yb = torch.tensor(y[j], device=device)
        yield Xb, yb

# >>> FIX: instanciar el MLP con NUM_CLASSES correcto
mlp = MLPBaseline(X_train.shape[1], num_classes=NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(mlp.parameters(), lr=2e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

EPOCHS_BASELINE = 5
best_val_acc = -1.0
best_mlp_state = None

for epoch in range(1, EPOCHS_BASELINE + 1):
    mlp.train()
    total_loss = 0.0
    for Xb, yb in iter_loader_sparse(X_train, y_train, batch_size=256, shuffle=True, device=DEVICE):
        optimizer.zero_grad()
        logits = mlp(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validación
    mlp.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for Xb, yb in iter_loader_sparse(X_val, y_val, batch_size=512, shuffle=False, device=DEVICE):
            logits = mlp(Xb)
            pred = logits.argmax(dim=-1)
            all_preds.append(pred.cpu().numpy())
            all_true.append(yb.cpu().numpy())

    y_pred_val = np.concatenate(all_preds)
    y_true_val = np.concatenate(all_true)
    val_acc = accuracy_score(y_true_val, y_pred_val)
    val_f1  = f1_score(y_true_val, y_pred_val, average="macro")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # guardamos copia en CPU para poder recargar en cualquier dispositivo
        best_mlp_state = {k: v.detach().cpu().clone() for k, v in mlp.state_dict().items()}

    print(f"[MLP] Epoch {epoch}/{EPOCHS_BASELINE} | TrainLoss={total_loss:.3f} | ValAcc={val_acc:.4f} | ValF1={val_f1:.4f}")

# Test con el mejor estado
mlp.load_state_dict(best_mlp_state)
mlp.to(DEVICE)
mlp.eval()
with torch.no_grad():
    all_preds = []
    for Xb, _ in iter_loader_sparse(X_test, y_test, batch_size=512, shuffle=False, device=DEVICE):
        logits = mlp(Xb)
        all_preds.append(logits.argmax(dim=-1).cpu().numpy())

y_pred_test_mlp = np.concatenate(all_preds)
mlp_acc_test = accuracy_score(y_test, y_pred_test_mlp)
mlp_f1_test  = f1_score(y_test, y_pred_test_mlp, average="macro")
print(f"[MLP] TestAcc={mlp_acc_test:.4f} | TestF1={mlp_f1_test:.4f}")

NUM_CLASSES = 5
[MLP] Epoch 1/5 | TrainLoss=62.454 | ValAcc=0.4905 | ValF1=0.4933
[MLP] Epoch 2/5 | TrainLoss=30.928 | ValAcc=0.4650 | ValF1=0.4657
[MLP] Epoch 3/5 | TrainLoss=10.439 | ValAcc=0.4455 | ValF1=0.4347
[MLP] Epoch 4/5 | TrainLoss=2.892 | ValAcc=0.4370 | ValF1=0.4401
[MLP] Epoch 5/5 | TrainLoss=1.088 | ValAcc=0.4360 | ValF1=0.4351
[MLP] TestAcc=0.4910 | TestF1=0.4946


TRANSFORMER

In [15]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
if tok.pad_token is None:
    tok.pad_token = tok.eos_token if tok.eos_token else "[PAD]"

MAX_LEN = 256
BATCH_SIZE_TX = 16 if torch.cuda.is_available() else 8

# Si no existe DEVICE en tu notebook:
try:
    DEVICE
except NameError:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Detecta número de clases a partir de tus labels (TR_Y/VAL_Y/TE_Y deben existir)
NUM_CLASSES = len(set(TR_Y) | set(VAL_Y) | set(TE_Y))
print("NUM_CLASSES =", NUM_CLASSES)  # debería imprimir 5 para SetFit/amazon_reviews_multi_es

# ----------------- Dataset/Dataloaders -----------------
from torch.utils.data import Dataset, DataLoader

class ReviewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tok = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = self.texts[idx]
        y = self.labels[idx]
        inputs = self.tok(text, truncation=True, padding="max_length", max_length=self.max_len)
        return {
            "input_ids": torch.tensor(inputs["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(inputs["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(y, dtype=torch.long)
        }

train_ds_tx = ReviewsDataset(TR_TEXTS, TR_Y, tok, MAX_LEN)
val_ds_tx   = ReviewsDataset(VAL_TEXTS, VAL_Y, tok, MAX_LEN)
test_ds_tx  = ReviewsDataset(TE_TEXTS, TE_Y, tok, MAX_LEN)

train_loader_tx = DataLoader(train_ds_tx, batch_size=BATCH_SIZE_TX, shuffle=True,  drop_last=False)
val_loader_tx   = DataLoader(val_ds_tx,   batch_size=BATCH_SIZE_TX, shuffle=False, drop_last=False)
test_loader_tx  = DataLoader(test_ds_tx,  batch_size=BATCH_SIZE_TX, shuffle=False, drop_last=False)

# ----------------- Positional Encoding -----------------
import math
import torch.nn.functional as F

class SinusoidalPE(torch.nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)
        i   = torch.arange(d_model).unsqueeze(0)
        div = torch.pow(10000, (2 * (i // 2)) / torch.tensor(d_model, dtype=torch.float32))
        angle = pos / div
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(angle[:, 0::2])
        pe[:, 1::2] = torch.cos(angle[:, 1::2])
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# ----------------- Multi-Head Attention -----------------
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim debe ser divisible por num_heads"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.Wq = torch.nn.Linear(embed_dim, embed_dim)
        self.Wk = torch.nn.Linear(embed_dim, embed_dim)
        self.Wv = torch.nn.Linear(embed_dim, embed_dim)
        self.Wo = torch.nn.Linear(embed_dim, embed_dim)

    def _split_heads(self, x):
        b, s, e = x.size()
        x = x.view(b, s, self.num_heads, self.head_dim)
        return x.permute(0, 2, 1, 3)

    def forward(self, x, mask=None):
        Q = self._split_heads(self.Wq(x))
        K = self._split_heads(self.Wk(x))
        V = self._split_heads(self.Wv(x))
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            m = mask.unsqueeze(1).unsqueeze(2)  # (b,1,1,seq)
            scores = scores.masked_fill(m == 0, -1e9)
        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)
        context = context.permute(0, 2, 1, 3).contiguous()
        context = context.view(context.size(0), context.size(1), -1)
        return self.Wo(context)

# ----------------- Bloque Transformer -----------------
class TransformerBlock(torch.nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim=512, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(embed_dim, num_heads)
        self.ln1 = torch.nn.LayerNorm(embed_dim)
        self.ff  = torch.nn.Sequential(
            torch.nn.Linear(embed_dim, ff_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(ff_dim, embed_dim)
        )
        self.ln2 = torch.nn.LayerNorm(embed_dim)
        self.drop = torch.nn.Dropout(dropout)

    def forward(self, x, mask):
        h = self.mha(x, mask=mask)
        x = self.ln1(x + self.drop(h))
        h2 = self.ff(x)
        x = self.ln2(x + self.drop(h2))
        return x

# ----------------- Clasificador completo -----------------
class SimpleTransformerClassifier(torch.nn.Module):
    def __init__(self, vocab_size, num_classes, max_len=256, embed_dim=256, num_heads=8, depth=2, dropout=0.1, pad_id=0):
        super().__init__()
        self.embed = torch.nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.pos   = SinusoidalPE(max_len, embed_dim)
        self.blocks = torch.nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim=4*embed_dim, dropout=dropout) for _ in range(depth)
        ])
        self.norm = torch.nn.LayerNorm(embed_dim)
        self.cls_head = torch.nn.Sequential(
            torch.nn.Linear(embed_dim, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(256, num_classes)
        )

    @staticmethod
    def masked_mean(x, mask):
        mask = mask.unsqueeze(-1)        # (b,s,1)
        x = x * mask
        denom = mask.sum(dim=1).clamp(min=1)
        return x.sum(dim=1) / denom

    def forward(self, input_ids, attention_mask):
        x = self.embed(input_ids)        # (b,s,d)
        x = self.pos(x)
        for blk in self.blocks:
            x = blk(x, mask=attention_mask)
        x = self.norm(x)
        pooled = self.masked_mean(x, attention_mask)
        return self.cls_head(pooled)

# Instanciar con el número correcto de clases
vocab_size = tok.vocab_size
pad_id = tok.pad_token_id if tok.pad_token_id is not None else 0
model_tx = SimpleTransformerClassifier(
    vocab_size=vocab_size,
    num_classes=NUM_CLASSES,   # <<< importante
    max_len=MAX_LEN,
    embed_dim=256,
    num_heads=8,
    depth=2,
    dropout=0.1,
    pad_id=pad_id
).to(DEVICE)

optimizer_tx = torch.optim.AdamW(model_tx.parameters(), lr=2e-4, weight_decay=1e-4)
criterion_tx = nn.CrossEntropyLoss()

def evaluate_model(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            y   = batch["labels"].to(DEVICE)
            logits = model(ids, mask)
            preds = logits.argmax(dim=-1)
            y_true.append(y.cpu().numpy()); y_pred.append(preds.cpu().numpy())
    yt = np.concatenate(y_true); yp = np.concatenate(y_pred)
    return accuracy_score(yt, yp), f1_score(yt, yp, average="macro")

EPOCHS_TX = 5
best_val_acc_tx = -1.0
best_state_tx = None
patience = 2
epochs_no_improve = 0

for epoch in range(1, EPOCHS_TX+1):
    model_tx.train()
    total_loss = 0.0
    for batch in train_loader_tx:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y   = batch["labels"].to(DEVICE)

        optimizer_tx.zero_grad()
        logits = model_tx(ids, mask)
        loss = criterion_tx(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_tx.parameters(), max_norm=1.0)
        optimizer_tx.step()
        total_loss += loss.item()

    val_acc, val_f1 = evaluate_model(model_tx, val_loader_tx)
    print(f"[TX] Epoch {epoch}/{EPOCHS_TX} | TrainLoss={total_loss:.3f} | ValAcc={val_acc:.4f} | ValF1={val_f1:.4f}")

    if val_acc > best_val_acc_tx:
        best_val_acc_tx = val_acc
        best_state_tx = {k: v.cpu().clone() for k, v in model_tx.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("[TX] Early stopping por paciencia")
            break

# Test con el mejor estado
model_tx.load_state_dict(best_state_tx)
model_tx.to(DEVICE)
tx_acc_test, tx_f1_test = evaluate_model(model_tx, test_loader_tx)
print(f"[TX] TestAcc={tx_acc_test:.4f} | TestF1={tx_f1_test:.4f}")


NUM_CLASSES = 5
[TX] Epoch 1/5 | TrainLoss=2121.960 | ValAcc=0.4010 | ValF1=0.3887
[TX] Epoch 2/5 | TrainLoss=1857.796 | ValAcc=0.4485 | ValF1=0.4263
[TX] Epoch 3/5 | TrainLoss=1700.139 | ValAcc=0.4560 | ValF1=0.4513
[TX] Epoch 4/5 | TrainLoss=1565.355 | ValAcc=0.4275 | ValF1=0.4023
[TX] Epoch 5/5 | TrainLoss=1390.883 | ValAcc=0.4405 | ValF1=0.4323
[TX] Early stopping por paciencia
[TX] TestAcc=0.4530 | TestF1=0.4473


Comparación y conclusiones

In [17]:
print(f"Baseline MLP (TF-IDF):   TestAcc={mlp_acc_test:.4f} | TestF1={mlp_f1_test:.4f}")
print(f"Transformer (from-scratch): TestAcc={tx_acc_test:.4f} | TestF1={tx_f1_test:.4f}")

conclusiones = f"""
CONCLUSIONES (breves y accionables):
1) Reproducibilidad: fijamos semillas, splits estratificados y parámetro MAX_LEN= {MAX_LEN}.
   Todas las celdas son ejecutables de cero y los resultados se pueden replicar.

2) Baseline MLP (TF-IDF): ofrece un punto de referencia sólido a bajo costo computacional.
   Métricas de test -> Acc={mlp_acc_test:.4f}, F1_macro={mlp_f1_test:.4f}.

3) Transformer "original": implementamos PE sinusoidal, Multi-Head Attention, bloques encoder,
   pooling enmascarado y cabeza de clasificación. Se entrena desde cero con ids de tokenizer XLM-R
   (solo usamos el tokenizador, no el modelo).
   Métricas de test -> Acc={tx_acc_test:.4f}, F1_macro={tx_f1_test:.4f}.

4) Comparación: cuando el dataset es variado y con dependencias contextuales, el Transformer
   captura mejor relaciones y suele superar o empatar al baseline. Si el baseline se acerca,
   suele indicar que n-gramas TF-IDF ya capturan suficiente señal (textos cortos, vocabulario directo).

5) Coste/recursos: el Transformer demanda más tiempo y VRAM; para asegurar reproducibilidad
   evitamos un clasificador por "flatten" y usamos pooling enmascarado, reduciendo drásticamente
   los parámetros y estabilizando el entrenamiento.

6) Mejoras futuras:
   - Aumentar épocas y/o tamaño de entrenamiento
   - Aumentar embed_dim o profundidad (con GPU)
   - Pre-entrenar embeddings o usar inicialización previsora
   - Regularización (dropout, weight decay) y ajuste fino de LR
   - Probar tokenizador entrenado en dominio o fine-tuning de un modelo preentrenado (p.ej., DistilBERT mBERT)
"""
print(conclusiones)

Baseline MLP (TF-IDF):   TestAcc=0.4910 | TestF1=0.4946
Transformer (from-scratch): TestAcc=0.4530 | TestF1=0.4473

CONCLUSIONES (breves y accionables):
1) Reproducibilidad: fijamos semillas, splits estratificados y parámetro MAX_LEN= 256. 
   Todas las celdas son ejecutables de cero y los resultados se pueden replicar.

2) Baseline MLP (TF-IDF): ofrece un punto de referencia sólido a bajo costo computacional.
   Métricas de test -> Acc=0.4910, F1_macro=0.4946.

3) Transformer "original": implementamos PE sinusoidal, Multi-Head Attention, bloques encoder, 
   pooling enmascarado y cabeza de clasificación. Se entrena desde cero con ids de tokenizer XLM-R
   (solo usamos el tokenizador, no el modelo). 
   Métricas de test -> Acc=0.4530, F1_macro=0.4473.

4) Comparación: cuando el dataset es variado y con dependencias contextuales, el Transformer 
   captura mejor relaciones y suele superar o empatar al baseline. Si el baseline se acerca, 
   suele indicar que n-gramas TF-IDF ya capturan 